In [1]:
import numpy as np
import gif
import iw
from tqdm.notebook import tqdm
import pandas as pd
import functions
import polars as pl

In [2]:
sessions, conversions, conversion_rates = iw.generate_dataset(
    num_users=10_000,
    sessions_skew=2.5,
    # cvr_func=lambda x,y: x*(1/10 + 9/10*np.exp(-0.05*(y-1))),
    cvr_decay_factor=0.1,
    beta_size=100,
    baseline_conversion_rate=0.2,
)

sessions_ctrl, sessions_test, conversions_ctrl, conversions_test = (
    iw.generate_experiment_datasets(sessions, conversions, num_experiments=10000)
)

In [3]:
# run all tests
bootstrapped_null_dist = iw.get_bootstrapped_null_hypothesis_distribution(
    sessions=np.append(sessions_ctrl[0], sessions_test[0]),
    conversions=np.append(conversions_ctrl[0], conversions_test[0]),
    num_bootstraps=10000,
)

bs_p_values = iw.bootstrapped_p_values(
    sessions_ctrl,
    conversions_ctrl,
    sessions_test,
    conversions_test,
    bootstrapped_null_dist,
)

binom_p_values = iw.binomial_z_test(
    sessions_ctrl, conversions_ctrl, sessions_test, conversions_test
)
delta_p_values = iw.delta_method_z_test(
    sessions_ctrl, conversions_ctrl, sessions_test, conversions_test
)

In [4]:
print((bs_p_values <= 0.05).mean())
print((delta_p_values <= 0.05).mean())

0.0605
0.0778


In [9]:
def datasets(source="ian", **kwargs):
    if source == "ian":
        sessions, conversions, conversion_rates = iw.generate_dataset(
            num_users=kwargs.get("num_users", 10_000),
            baseline_conversion_rate=kwargs.get("baseline_conversion_rate", 0.6),
            sessions_skew=kwargs.get("sessions_skew", 2.5),
            cvr_decay_factor=kwargs.get("cvr_decay_factor", 0.1),
            beta_size=kwargs.get("beta_size", 100),
        )

        return (sessions, conversion_rates),  iw.generate_experiment_datasets(
            sessions, conversions, num_experiments=kwargs.get("num_experiments", 1_000)
        )
    elif source == "functions":
        df = functions.generate_dataset(
            num_units=kwargs.get("num_units", 10_000),
            num_obs=kwargs.get("num_obs", 100_000),
            impact=kwargs.get("impact", 0.0),
            distr=kwargs.get("distr", "geom"),
            beta_size=kwargs.get("beta_size", 100),
            baseline_conversion_rate=kwargs.get("baseline_conversion_rate", 0.6),
        )
        return functions.convert_dataframe_to_experiment_arrays(df, num_experiments=kwargs.get("num_experiments", 1_000))

In [6]:
frames = []
frames_all = []
results = []

sessions_skew = 2.5
source = "functions"  # Change to "functions" to use functions.py dataset generation

for cvr_decay_factor in tqdm(np.linspace(0.0, 0.1, 10)):        
    (sessions, conversion_rates), (sessions_ctrl, sessions_test, conversions_ctrl, conversions_test) = datasets(
        source=source,
        sessions_skew=sessions_skew,        
        cvr_decay_factor=cvr_decay_factor,  
        num_units=60,
        num_users=100,
        num_obs=3000,      
    )

    # run all tests
    bootstrapped_null_dist = iw.get_bootstrapped_null_hypothesis_distribution(
        sessions=np.append(sessions_ctrl[0], sessions_test[0]),
        conversions=np.append(conversions_ctrl[0], conversions_test[0]),
        num_bootstraps=10000,
    )

    bs_p_values = iw.bootstrapped_p_values(
        sessions_ctrl,
        conversions_ctrl,
        sessions_test,
        conversions_test,
        bootstrapped_null_dist,
    )

    binom_p_values = iw.binomial_z_test(
        sessions_ctrl, conversions_ctrl, sessions_test, conversions_test
    )
    delta_p_values = iw.delta_method_z_test(
        sessions_ctrl, conversions_ctrl, sessions_test, conversions_test
    )

    results.append(
        {
            "cvr_decay_factor": cvr_decay_factor,
            "binom_p_values": np.mean(binom_p_values <= 0.05),
            "delta_p_values": np.mean(delta_p_values <= 0.05),
            "bs_p_values": np.mean(bs_p_values <= 0.05),
            "sessions_skew": sessions_skew,
        }
    )

    frame = iw.plot(
        sessions,
        conversion_rates,
        binom_p_values,
        delta_p_values,
        bs_p_values,
        cvr_decay_factor,
        plot_all=False,
    )
    frames.append(frame)

    frame = iw.plot(
        sessions,
        conversion_rates,
        binom_p_values,
        delta_p_values,
        bs_p_values,
        cvr_decay_factor,
        plot_all=True,
    )
    frames_all.append(frame)

  0%|          | 0/10 [00:00<?, ?it/s]

In [7]:
gif.save(frames, "p_values_cvr_decay_factor.gif", duration=1000)
gif.save(frames_all, "p_values_cvr_decay_factor_all.gif", duration=1000)

In [8]:
pd.DataFrame(results)

,cvr_decay_factor,binom_p_values,delta_p_values,bs_p_values,sessions_skew
0,0.000000,0.463,0.086,0.047,2.5
1,0.011111,0.527,0.078,0.053,2.5
2,0.022222,0.510,0.104,0.060,2.5
3,0.033333,0.461,0.074,0.055,2.5
4,0.044444,0.877,0.123,0.043,2.5
5,0.055556,0.484,0.069,0.041,2.5
6,0.066667,0.546,0.092,0.055,2.5
7,0.077778,0.641,0.118,0.066,2.5
8,0.088889,0.619,0.124,0.054,2.5
9,0.100000,0.666,0.098,0.045,2.5
